# Laboratorio 3.3 — Tu primer pipeline distribuido con Spark SQL

**Módulo 3 · Herramientas y Tecnologías** — bloque [`03-spark-y-procesamiento-distribuido.md`](../../../Apuntes-Markdown/03-herramientas-y-tecnologias/03-spark-y-procesamiento-distribuido.md)

**Duración orientativa:** 90 minutos · **Modalidad:** individual · **Herramientas:** Google Colab + PySpark

---

## Cómo usar este notebook

Todo el código de este notebook está ya escrito, revisado dos veces y verificado por otra vía (comparando cada resultado con el cálculo equivalente hecho en pandas sobre el mismo CSV, tal y como haréis vosotros en el laboratorio 3.1). Solo tenéis que ejecutarlo celda a celda, de arriba abajo, en **Google Colab** (necesitáis conexión a internet para instalar PySpark, así que este notebook no funciona en el entorno sin red que se usó para preparar el resto del curso).

## Objetivo de aprendizaje

Repetir en Spark SQL algunas de las consultas del laboratorio 3.2 y observar las diferencias de sintaxis frente a SQL/pandas tradicional, además de una primera introducción al plan de ejecución distribuido: particiones, shuffle y transformaciones narrow vs. wide.

## Contexto

En el laboratorio 3.2 trabajasteis con SQL "clásico" (SQLite) sobre una base de datos que cabe entera en la memoria de un único ordenador. El apunte del bloque 3 explica que Apache Spark existe precisamente para cuando los datos **no** caben en una sola máquina: reparte el dataset en particiones y ejecuta las transformaciones en paralelo sobre un clúster de nodos, usando **evaluación perezosa (lazy evaluation)** — las transformaciones no se ejecutan hasta que se solicita un resultado concreto (una action).

Spark SQL es la capa que permite escribir esas transformaciones distribuidas con sintaxis SQL estándar, registrando un DataFrame como una "vista temporal" (`createOrReplaceTempView`) y consultándolo con `spark.sql("...")`. La API resultante es casi idéntica a la que usasteis en 3.2, pero por debajo el motor construye un plan de ejecución distribuido (DAG) que el optimizador Catalyst reordena para minimizar el trabajo, especialmente el shuffle: la redistribución de datos entre nodos que provocan operaciones como `groupBy` o `join`.

Aquí usaréis `tienda_online_ventas.csv`, el mismo CSV desnormalizado del laboratorio 3.1 (no las tablas separadas de `tienda.db`), así que las consultas se adaptan a un único esquema plano en vez de a tablas relacionadas con `JOIN`.

## Dataset

`tienda_online_ventas.csv` (en esta misma carpeta), el mismo dataset de 12.000 pedidos que en 3.1 y 3.5.


## Paso 0 — Instalar PySpark y arrancar una sesión de Spark (10 min)

Google Colab no trae PySpark preinstalado, así que la primera celda lo instala con `pip`. Esto solo funciona con conexión a internet (Colab la tiene por defecto).

In [ ]:
# Esta celda solo funciona en un entorno con acceso a internet (como Google Colab).
!pip install pyspark --quiet

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("Laboratorio3.3-PipelineSparkSQL")
    .master("local[*]")  # usa todos los núcleos disponibles de la máquina de Colab como "clúster" local
    .getOrCreate()
)

print("Sesión de Spark creada. Versión:", spark.version)

## Paso 1 — Cargar el CSV en un DataFrame de Spark (10 min)

Usamos `spark.read.csv()` con `header=True` (la primera fila son los nombres de columna) e `inferSchema=True` (Spark detecta automáticamente el tipo de cada columna, en vez de tratarlas todas como texto).

In [ ]:
ventas_spark = spark.read.csv("tienda_online_ventas.csv", header=True, inferSchema=True)

print(f"Número de particiones del DataFrame: {ventas_spark.rdd.getNumPartitions()}")
ventas_spark.printSchema()

In [ ]:
# show() es una action: dispara la ejecución real y materializa el resultado.
ventas_spark.show(5)

In [ ]:
# count() es otra action habitual: cuenta el número de filas.
print("Número total de filas:", ventas_spark.count())

## Paso 2 — Registrar el DataFrame como vista temporal (5 min)

`createOrReplaceTempView()` no mueve ni copia datos: simplemente le da un nombre SQL al DataFrame para poder consultarlo con `spark.sql("...")`, como si fuera una tabla.

In [ ]:
ventas_spark.createOrReplaceTempView("ventas")
print("Vista temporal 'ventas' registrada.")

## Paso 3 — Repetir 5 consultas del laboratorio 3.2 en Spark SQL (30 min)

Adaptamos las consultas al esquema **desnormalizado** de este CSV (aquí no hay tablas separadas de clientes/productos: todo vive en una única tabla `ventas`, así que no hace falta ningún `JOIN` para las que en 3.2 sí lo necesitaban).

Cada resultado ha sido verificado antes de escribir esta celda calculando el mismo número con pandas sobre el mismo CSV (podéis comprobarlo vosotros mismos repitiendo el Paso 4 del laboratorio 3.1).

### Consulta 1 — Ventas totales, medias y número de pedidos por categoría

Equivalente a la Consulta 6 del laboratorio 3.2, pero sin necesidad de `JOIN` porque `categoria` ya está en la misma tabla.

In [ ]:
resultado_1 = spark.sql('''
    SELECT categoria,
           ROUND(SUM(importe), 2) AS importe_total,
           ROUND(AVG(importe), 2) AS importe_medio,
           COUNT(*) AS num_pedidos
    FROM ventas
    GROUP BY categoria
    ORDER BY importe_total DESC
''')
resultado_1.show()

# Verificación esperada (calculada con pandas sobre el mismo CSV):
# Electrónica  824029.78  257.51  3200
# Deporte      682886.96  281.37  2427
# Papelería    485748.23  237.76  2043
# Hogar        436042.83  186.34  2340
# Moda         333535.40  167.61  1990

### Consulta 2 — Top 10 clientes por gasto total

Equivalente a la Consulta 5 de 3.2. Aquí no hace falta `JOIN` con una tabla de clientes porque `cliente_nombre` ya está en cada fila.

In [ ]:
resultado_2 = spark.sql('''
    SELECT cliente_id, cliente_nombre,
           ROUND(SUM(importe), 2) AS gasto_total
    FROM ventas
    GROUP BY cliente_id, cliente_nombre
    ORDER BY gasto_total DESC
    LIMIT 10
''')
resultado_2.show()

# Verificación esperada: el cliente C100164 (Cliente 0165) encabeza el ranking con 12165.81 EUR.

### Consulta 3 — Ventas por región y canal

Agregación con dos claves de agrupación, igual que la Consulta 10 de 3.2 pero cruzando región en vez de método de pago.

In [ ]:
resultado_3 = spark.sql('''
    SELECT region, canal,
           ROUND(SUM(importe), 2) AS importe_total,
           COUNT(*) AS num_pedidos
    FROM ventas
    GROUP BY region, canal
    ORDER BY importe_total DESC
''')
resultado_3.show(10)

# Verificación esperada: la combinación Levante + Web encabeza con 434205.96 EUR.

### Consulta 4 — Regiones con importe medio por pedido superior a 200 EUR (HAVING)

Equivalente a la Consulta 7 de 3.2.

In [ ]:
resultado_4 = spark.sql('''
    SELECT region,
           ROUND(AVG(importe), 2) AS importe_medio,
           ROUND(SUM(importe), 2) AS importe_total
    FROM ventas
    GROUP BY region
    HAVING AVG(importe) > 200
    ORDER BY importe_medio DESC
''')
resultado_4.show()

# Verificación esperada: las 5 regiones superan los 200 EUR de media (Cataluña la más alta, 238.71).

### Consulta 5 — Ranking de clientes por gasto total (window function RANK)

Equivalente a la Consulta 8 de 3.2. Spark SQL soporta window functions con la misma sintaxis estándar que SQLite.

In [ ]:
resultado_5 = spark.sql('''
    SELECT cliente_id, cliente_nombre, gasto_total,
           RANK() OVER (ORDER BY gasto_total DESC) AS ranking_gasto
    FROM (
        SELECT cliente_id, cliente_nombre, ROUND(SUM(importe), 2) AS gasto_total
        FROM ventas
        GROUP BY cliente_id, cliente_nombre
    )
    ORDER BY ranking_gasto
    LIMIT 10
''')
resultado_5.show()

# Verificación esperada: mismo top 10 y mismos importes que en la Consulta 2, con la columna
# adicional 'ranking_gasto' (1 a 10).

## Paso 4 — La misma consulta con la API de DataFrame (10 min)

Spark ofrece dos formas equivalentes de expresar transformaciones: SQL (`spark.sql`) o la **API de DataFrame** (`.filter()`, `.groupBy()`, `.agg()`). Repetimos la Consulta 1 (ventas por categoría) con la API de DataFrame.

In [ ]:
resultado_1_api = (
    ventas_spark
    .groupBy("categoria")
    .agg(
        F.round(F.sum("importe"), 2).alias("importe_total"),
        F.round(F.avg("importe"), 2).alias("importe_medio"),
        F.count("*").alias("num_pedidos"),
    )
    .orderBy(F.col("importe_total").desc())
)
resultado_1_api.show()

# Debe producir exactamente el mismo resultado que resultado_1 (Consulta 1 en SQL).

**Comparación de estilos SQL vs. API de DataFrame:**

Ambas expresiones (`spark.sql("SELECT categoria, SUM(importe) ... GROUP BY categoria")` y `ventas_spark.groupBy("categoria").agg(F.sum("importe")...)`) generan **el mismo plan de ejecución** por debajo: Spark SQL compila la consulta textual a la misma representación interna que usa la API de DataFrame, así que el rendimiento es idéntico. La elección es puramente de estilo: SQL resulta más natural para quien viene de bases de datos relacionales (como en el laboratorio 3.2), mientras que la API de DataFrame se integra mejor cuando el pipeline mezcla lógica Python (bucles, funciones condicionales, listas de columnas generadas dinámicamente) con las transformaciones de datos.

## Paso 5 — Observar el plan de ejecución con .explain() (15 min)

`.explain()` muestra el plan físico que Spark va a ejecutar. Buscamos la palabra **"Exchange"** en el plan: indica que Spark necesita un **shuffle**, es decir, redistribuir datos entre particiones (y potencialmente entre nodos del clúster) para completar la operación.

In [ ]:
print("### Plan de ejecución de una transformación WIDE (groupBy): ###")
resultado_1_api.explain()

# Buscad la línea que contiene "Exchange hashpartitioning" en la salida: es la señal de un shuffle.
# El groupBy necesita reunir en la misma partición todas las filas con la misma 'categoria'
# antes de poder sumar, y esas filas pueden estar repartidas en particiones distintas: de ahí
# el Exchange.

In [ ]:
print("### Plan de ejecución de una transformación NARROW (filter): ###")
solo_electronica = ventas_spark.filter(F.col("categoria") == "Electrónica")
solo_electronica.explain()

# En este plan NO debería aparecer "Exchange": un filter() se puede aplicar de forma
# completamente independiente dentro de cada partición, sin necesitar datos de otras
# particiones. Por eso es una transformación 'narrow'.

**Nota de una línea sobre shuffle:** la Consulta 1 (`groupBy("categoria")`) genera shuffle porque es una transformación *wide* — necesita reunir todas las filas de una misma categoría en la misma partición antes de agregarlas —, mientras que el `filter()` del Paso 5 no genera shuffle porque es una transformación *narrow*, que cada partición puede resolver de forma completamente local sin comunicarse con las demás.

Esto conecta directamente con el apunte del bloque 3: las operaciones que requieren shuffle (`groupBy`, `join`, `orderBy` global) son las más costosas de un pipeline Spark porque implican serialización, transferencia de red y E/S a disco entre nodos, así que minimizarlas (o beneficiarse de optimizaciones como el *broadcast join* cuando una tabla es pequeña) es clave para el rendimiento.

## Cierre

Cerramos la sesión de Spark al terminar, como buena práctica (libera los recursos del clúster local).

In [ ]:
spark.stop()
print("Sesión de Spark cerrada.")

## Entregable

Este notebook ejecutado de principio a fin en Google Colab, con las 5 consultas replicadas en Spark SQL, la Consulta 1 repetida con la API de DataFrame, y la nota de una línea sobre cuál de las operaciones observadas generó shuffle y por qué (Paso 5).

## Preguntas de reflexión

1. Todas las consultas de este notebook usan `tienda_online_ventas.csv` en lugar de las tablas normalizadas de `tienda.db`. Si tuvierais que ejecutar en Spark SQL una consulta con `JOIN` como la Consulta 5 del laboratorio 3.2 (clientes + pedidos + detalle_pedido), ¿qué tabla creeríais que sería buena candidata para un *broadcast join* del apunte, y por qué? (Pista: fijaos en cuántas filas tiene cada tabla).
2. ¿Por qué `spark.read.csv(...)` **no** dispara ninguna acción real (no lee todavía todos los datos línea a línea), mientras que `ventas_spark.count()` sí lo hace? Relacionadlo con el concepto de *lazy evaluation* del apunte.
3. Si este dataset tuviera 500 millones de filas en lugar de 12.000, ¿cambiaría alguna de las consultas SQL que habéis escrito? ¿Y el número de particiones que le convendría tener al DataFrame?
